# W8C2 Lab: Measuring a Scaling Law

Run every cell from the top. **Everything already works.**

Today you will:

1. Train five models of increasing size on the same task.
2. Plot loss against size and find the straight line.
3. See what more data does that more parameters cannot.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup. A small character-level task, so five models train in about a minute.
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)

# Varied text, not one sentence on a loop. If the text repeats, every model
# memorises it and the experiment measures nothing.
TEXT = ("machine learning models learn patterns from data and then make predictions "
        "about examples they have never seen before which is called generalisation "
        "a language model reads text and predicts what word is likely to come next "
        "training adjusts millions of numbers so that the predictions become better "
        "researchers measure quality with a loss that falls as the model improves "
        "small models underfit because they cannot represent the patterns in the data "
        "large models can overfit when there is not enough data to constrain them "
        "the balance between capacity and data is the central question of scaling ") * 8

chars = sorted(set(TEXT))
stoi = {c: i for i, c in enumerate(chars)}
full = torch.tensor([stoi[c] for c in TEXT])

# HELD-OUT split. Loss is always measured on text the model never trained on,
# otherwise a big model just memorises and looks better than it is.
cut = int(len(full) * 0.8)
train_data, val_data = full[:cut], full[cut:]

CONTEXT = 16
def batches(source, n=64):
    starts = torch.randint(0, len(source) - CONTEXT - 1, (n,))
    x = torch.stack([source[s:s + CONTEXT] for s in starts])
    y = torch.stack([source[s + 1:s + CONTEXT + 1] for s in starts])
    return x, y

print(f"{len(TEXT):,} characters, {len(chars)} distinct")
print(f"{len(train_data):,} for training, {len(val_data):,} held back for scoring")

## Part 1. Train five models, change only the size

Everything is held fixed except the width of the hidden layer. That is the
experiment: does a bigger model get a lower loss, and by how much?

In [ ]:
# GIVEN. One model definition, five sizes. Only the width changes.
class Tiny(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.embed = nn.Embedding(len(chars), 16)
        self.rnn = nn.GRU(16, hidden, batch_first=True)
        self.out = nn.Linear(hidden, len(chars))

    def forward(self, x):
        h, _ = self.rnn(self.embed(x))
        return self.out(h)

loss_fn = nn.CrossEntropyLoss()

def train(hidden, steps=400, source=train_data):
    """Train one model, then score it on the HELD-OUT text."""
    torch.manual_seed(0)
    m = Tiny(hidden)
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    for _ in range(steps):
        x, y = batches(source)
        opt.zero_grad()
        loss_fn(m(x).reshape(-1, len(chars)), y.reshape(-1)).backward()
        opt.step()
    with torch.no_grad():
        x, y = batches(val_data, 256)
        val = loss_fn(m(x).reshape(-1, len(chars)), y.reshape(-1)).item()
    return sum(p.numel() for p in m.parameters()), val

SIZES = [4, 8, 16, 32, 64]
results = [train(h) for h in SIZES]

print(f"{'hidden':>7} {'parameters':>11} {'held-out loss':>14}")
for h, (params, loss) in zip(SIZES, results):
    print(f"{h:>7} {params:>11,} {loss:>14.3f}")

In [ ]:
# GIVEN. The scaling plot. Both axes are logarithmic, which is the trick.
params = [p for p, l in results]
losses = [l for p, l in results]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.4))
a1.plot(params, losses, "o-", color="#7C2529")
a1.set_xlabel("parameters"); a1.set_ylabel("loss"); a1.set_title("linear axes")

a2.loglog(params, losses, "o-", color="#7C2529")
a2.set_xlabel("parameters (log)"); a2.set_ylabel("loss (log)")
a2.set_title("log-log axes: a straight line")
plt.tight_layout(); plt.show()

slope = np.polyfit(np.log(params), np.log(losses), 1)[0]
print(f"fitted slope on the log-log plot: {slope:.3f}")
print("A straight line on log-log axes IS a power law. That is the claim in")
print("Kaplan 2020 and Chinchilla, measured here on a model you can train.")

In [ ]:
# ================== YOUR TURN 1 ==================
# Add a much bigger model and see whether the line continues.
#
# Set EXTRA_HIDDEN to 128, then 256.
#
# Expected: the sweep runs 2.01, 1.50, 0.68, 0.34, 0.25, so doubling the model
#           does NOT halve the loss: it buys a fixed fraction, which is what a power
#           law means. At hidden 128 you get 0.26, very slightly WORSE than 64. The
#           curve has flattened: this task is small enough that the model has run out
#           of things to learn, and more parameters stop helping.
# ===============================================
EXTRA_HIDDEN = 128          # <-- try 256 as well

extra_params, extra_loss = train(EXTRA_HIDDEN)
all_params = params + [extra_params]
all_losses = losses + [extra_loss]

print(f"hidden {EXTRA_HIDDEN}: {extra_params:,} parameters, held-out loss {extra_loss:.3f}")
print(f"best previous:        {params[-1]:,} parameters, held-out loss {losses[-1]:.3f}")

plt.figure(figsize=(5.5, 3.4))
plt.loglog(params, losses, "o-", color="#999", label="original sweep")
plt.loglog([extra_params], [extra_loss], "o", color="#7C2529", markersize=10, label="your model")
plt.xlabel("parameters (log)"); plt.ylabel("loss (log)"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 2 ==================
# Now hold the model fixed and starve it of data instead.
#
# DATA_FRACTION cuts the training text. Try 0.1, then 0.5, then 1.0.
#
# Expected: at 10% the big model scores about 2.15 against the small model's
#           1.50, so bigger LOSES. It has enough capacity to memorise the fragment it
#           was given and nothing left over for text it has not seen. Parameters and
#           data have to grow together, which is the correction Chinchilla made.
# ===============================================
DATA_FRACTION = 0.1          # <-- try 0.5, then 1.0

subset = train_data[:int(len(train_data) * DATA_FRACTION)]
big_params, big_loss = train(64, source=subset)
small_params, small_loss = train(8, source=train_data)

print(f"data used: {DATA_FRACTION:.0%}  ({len(subset):,} characters)")
print(f"   BIG model (hidden 64) on this much data : held-out loss {big_loss:.3f}")
print(f"   small model (hidden 8) on ALL the data  : held-out loss {small_loss:.3f}")
print()
print("bigger model wins:", big_loss < small_loss)

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   The gain per doubling shrinks, and by hidden 128 it has stopped: 0.259
#   against 0.248 for hidden 64. A power law says loss falls by a constant
#   FACTOR for each constant factor of extra parameters, but that only holds
#   while there is signal left in the data. Here the task runs out first,
#   which is a small version of exactly what Chinchilla measured.
#
# YOUR TURN 2
#   At DATA_FRACTION = 0.1 the big model usually loses to the small one, and by
#   1.0 it wins comfortably. Capacity you cannot feed is wasted. Chinchilla's
#   result was exactly this: for a fixed compute budget, the models people were
#   training were far too big for the data they were given.